# Bölüm 3 — Bonus: Prompt iyileştirme (önce/sonra)

3.2'de bulunan hatalara (ironi, çift olumsuz, genel ifadelerden konu çıkarma, çekingen ifadeler) yönelik ikinci bir prompt. Önce: Prompt 1 (`bolum3_llm.ipynb`), sonra: Prompt 2. İkisi aynı model ve ayarlarla, aynı altın setle karşılaştırılıyor.
Girdi: `ciktilar/orneklem_200.csv`, `altin_etiket.csv`, `llm_etiketleri.csv` (bolum3_llm.ipynb'de üretildi).

In [1]:
import os, json, time, getpass, warnings
import numpy as np
import pandas as pd
from groq import Groq
warnings.filterwarnings('ignore', category=RuntimeWarning)

CIKTI = '../ciktilar/'
LLM_YENIDEN = False        # True: Prompt 2 yeniden çalıştırılır
SADECE_FARKLI_METIN = True # True: 35 farklı metin bir kez etiketlenir (kota için); False: 200 yorumun her biri ayrı istek

# Önceki notebook'ta üretilen dosyalar
orneklem = pd.read_csv(CIKTI + 'orneklem_200.csv')
altin = pd.read_csv(CIKTI + 'altin_etiket.csv', encoding='utf-8-sig').fillna('')
llm_1 = pd.read_csv(CIKTI + 'llm_etiketleri.csv', encoding='utf-8-sig').fillna('')
print('Örneklem:', len(orneklem), '| Altın:', len(altin), '| Prompt 1:', len(llm_1))

# Groq (Prompt 1 ile aynı model ve ayarlar: temperature=0, varsayılan akıl yürütme)
if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass.getpass('Groq API anahtarı: ')
groq_client = Groq()
MODEL = 'openai/gpt-oss-20b'

# Prompt 2: önce konu bazlı değerlendirmeleri çıkar, duyguyu bunlardan türet
# Örnekler veride OLMAYAN uydurma cümlelerden; test metinlerine özgü ipucu yok
PROMPT_2 = """Bir otomotiv yetkili servisine gelen müşteri yorumunu analiz edeceksin. Amaç, müşterinin \
neyi nasıl değerlendirdiğini çıkarmak.

Şu adımları izle:

1. Yorumda müşterinin değerlendirdiği her şeyi bul ve her biri için yönünü belirle (olumlu / olumsuz).
   Değerlendirilen şeyi şu gruplardan birine ata:
   randevu, fiyat, süre, personel, iş_kalitesi, yedek_parça, temizlik.
   Hiçbirine uymuyorsa ama somut bir şeyse "diğer" de. Yorumda geçmeyen bir şeyi ekleme.

2. Yönü belirlerken yüzeydeki kelimelere değil, müşterinin gerçekte ne demek istediğine bak.
   İnsanlar alay edebilir ya da olumsuz kalıplarla olumlu bir şey söyleyebilir.

3. Genel duyguyu bu listeden çıkar:
   - Sadece olumlu değerlendirmeler varsa → Olumlu
   - Sadece olumsuz değerlendirmeler varsa → Olumsuz
   - İkisi birden varsa → Karışık
   - Somut bir şey değerlendirilmiyorsa, yorumun genel tonuna bak: memnuniyet → Olumlu, şikâyet → Olumsuz,
     ne memnun ne şikâyetçi (çekingen, kararsız) → Karışık
   - Yorum hizmetle hiç ilgili değilse (anlamsız işaret gibi) → Alakasız

## Örnekler
Yorum: "Ne güzel, aracımı teslim aldığım gün yine arıza lambası yandı."
{{"degerlendirmeler": [{{"konu": "iş_kalitesi", "yon": "olumsuz"}}], "duygu": "Olumsuz", "konular": ["iş_kalitesi"]}}

Yorum: "Fiyat makuldü ama servis alanı oldukça kirliydi."
{{"degerlendirmeler": [{{"konu": "fiyat", "yon": "olumlu"}}, {{"konu": "temizlik", "yon": "olumsuz"}}], "duygu": "Karışık", "konular": ["fiyat", "temizlik"]}}

Yorum: "Aldığım hizmetten pişman değilim."
{{"degerlendirmeler": [], "duygu": "Olumlu", "konular": []}}

Yorum: "Servisin girişinde engelli rampası yok."
{{"degerlendirmeler": [{{"konu": "diğer", "yon": "olumsuz"}}], "duygu": "Olumsuz", "konular": ["diğer"]}}

Sadece bu biçimde JSON döndür, başka hiçbir şey yazma.

Yorum: \"\"\"{metin}\"\"\""""


def llm_etiketle(metin, prompt, deneme=5):
    """Tek bir metni etiketler. JSON zorunlu, temperature=0; geçici hatalarda bekleyip tekrar dener."""
    bekleme = 5
    for i in range(deneme):
        try:
            cevap = groq_client.chat.completions.create(
                model=MODEL, messages=[{'role': 'user', 'content': prompt.format(metin=metin)}],
                temperature=0, response_format={'type': 'json_object'})
            ham = cevap.choices[0].message.content
            return ham, json.loads(ham)
        except Exception as e:
            if i < deneme - 1:
                print(f'  Hata ({type(e).__name__}): {str(e)[:150]} — {bekleme} sn bekleyip tekrar deneniyor...')
                time.sleep(bekleme)
                bekleme *= 2
            else:
                return f'HATA: {e}', None

Örneklem: 200 | Altın: 35 | Prompt 1: 200


In [2]:
# Prompt 2. Kayıtlı ve tam bir sonuç varsa LLM'e gidilmez; yarım kaldıysa kaldığı yerden devam eder
yol = CIKTI + 'llm_etiketleri_v2.csv'
onceki = pd.read_csv(yol, encoding='utf-8-sig').fillna('') if os.path.exists(yol) and not LLM_YENIDEN else None

if SADECE_FARKLI_METIN:
    # Her farklı metin bir kez etiketlenir, sonuç 200 yorumdaki tekrarlarına uygulanır (35 istek)
    gorevler = list(orneklem['metin'].unique())
    anahtar = 'metin'
else:
    # 200 yorumun her biri ayrı istek (Prompt 1 ile aynı yöntem)
    gorevler = list(orneklem['geri_bildirim_id'])
    anahtar = 'geri_bildirim_id'

tamam = {} if onceki is None else {k: r for k, r in onceki.set_index(anahtar)[['duygu', 'konular', 'ham_cevap']]
                                     .to_dict('index').items() if r['duygu']} if anahtar == 'geri_bildirim_id' \
        else ({} if onceki is None else onceki.drop_duplicates('metin').set_index('metin')[['duygu', 'konular', 'ham_cevap']]
              .to_dict('index'))
kalan = [g for g in gorevler if g not in tamam]
print(f'{len(gorevler) - len(kalan)} tamam, {len(kalan)} kaldı')

metin_bul = dict(zip(orneklem['geri_bildirim_id'], orneklem['metin']))
for i, g in enumerate(kalan, start=1):
    metin = g if anahtar == 'metin' else metin_bul[g]
    ham, sonuc = llm_etiketle(metin, PROMPT_2)
    if sonuc is None:
        print('Durduruldu, son hata:', ham[:300])
        break
    tamam[g] = {'duygu': sonuc.get('duygu', ''), 'konular': ', '.join(sonuc.get('konular', [])), 'ham_cevap': ham}
    if i % 5 == 0:
        print(f'{i}/{len(kalan)}')
    time.sleep(8)   # dakikadaki token sınırı (8.000) için

# 200 satırlık dosya (bitmeyenler boş kalır; tekrar çalıştırınca tamamlanır)
llm_2 = orneklem[['geri_bildirim_id', 'arac_id', 'metin']].copy()
for alan in ['duygu', 'konular', 'ham_cevap']:
    llm_2[alan] = llm_2[anahtar].map(lambda k: tamam.get(k, {}).get(alan, ''))
llm_2.to_csv(yol, index=False, encoding='utf-8-sig')
print('Kaydedildi. Etiketli satır:', (llm_2['duygu'] != '').sum(), '/ 200')

0 tamam, 35 kaldı
5/35
10/35
15/35
20/35
25/35
  Hata (BadRequestError): Error code: 400 - {'error': {'message': "Failed to generate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'inva — 5 sn bekleyip tekrar deneniyor...
30/35
35/35
Kaydedildi. Etiketli satır: 200 / 200


In [3]:
# Önce (Prompt 1) / sonra (Prompt 2): aynı altın set, aynı 40 kayıt ve aynı 200 kayıt
def birlestir(llm_df):
    b = llm_df.merge(altin[['metin', 'duygu', 'konular']], on='metin', suffixes=('_llm', '_altin'))
    kume = lambda k: frozenset(x.strip() for x in str(k).split(',') if x.strip())
    b['konu_llm'], b['konu_altin'] = b['konular_llm'].apply(kume), b['konular_altin'].apply(kume)
    b['duygu_dogru'] = b['duygu_llm'] == b['duygu_altin']
    b['konu_dogru'] = b['konu_llm'] == b['konu_altin']
    return b

b1, b2 = birlestir(llm_1), birlestir(llm_2)

# Vakanın 40'lık altın seti: bolum3_llm.ipynb'deki seçimle aynı (aynı satırlar, aynı random_state)
altin_set_id = b1.sample(n=40, random_state=42)['geri_bildirim_id']

def ozet(b):
    s40 = b['geri_bildirim_id'].isin(altin_set_id)
    return {'Duygu, 40': b.loc[s40, 'duygu_dogru'].mean(), 'Duygu, 200': b['duygu_dogru'].mean(),
            'Konu tam eşleşme, 40': b.loc[s40, 'konu_dogru'].mean(), 'Konu tam eşleşme, 200': b['konu_dogru'].mean()}

tablo = pd.DataFrame({'Prompt 1 (önce)': ozet(b1), 'Prompt 2 (sonra)': ozet(b2)})
tablo['fark'] = tablo['Prompt 2 (sonra)'] - tablo['Prompt 1 (önce)']
print((tablo * 100).round(1).to_string())

# Metin bazında: hangi metinler düzeldi, hangileri bozuldu? (her metnin çoğunluk cevabıyla)
def metin_ozeti(b, sutun):
    return b.groupby('metin').agg(dogru=(sutun, 'mean'), llm=(sutun.replace('_dogru', '_llm') if 'duygu' in sutun
                                                              else 'konular_llm', lambda x: x.mode().iat[0]))

for alan, sutun, altin_sutun in [('DUYGU', 'duygu_dogru', 'duygu'), ('KONU', 'konu_dogru', 'konular')]:
    m1, m2 = metin_ozeti(b1, sutun), metin_ozeti(b2, sutun)
    k = m1.join(m2, lsuffix='_1', rsuffix='_2').join(altin.set_index('metin')[[altin_sutun]])
    k['durum'] = np.select([(k['dogru_1'] < 0.5) & (k['dogru_2'] >= 0.5), (k['dogru_1'] >= 0.5) & (k['dogru_2'] < 0.5),
                            (k['dogru_1'] < 0.5) & (k['dogru_2'] < 0.5)], ['DÜZELDİ', 'BOZULDU', 'hâlâ yanlış'], '')
    print(f'\n=== {alan} ===')
    print(k[k['durum'] != ''][['durum', altin_sutun, 'llm_1', 'llm_2']]
          .rename(columns={altin_sutun: 'altın', 'llm_1': 'Prompt 1', 'llm_2': 'Prompt 2'}).sort_values('durum').to_string())

                       Prompt 1 (önce)  Prompt 2 (sonra)  fark
Duygu, 40                         92.5              95.0   2.5
Duygu, 200                        89.0              91.0   2.0
Konu tam eşleşme, 40              62.5              67.5   5.0
Konu tam eşleşme, 200             73.5              72.5  -1.0

=== DUYGU ===
                                                                                      durum    altın Prompt 1 Prompt 2
metin                                                                                                                 
Arıza çözüldü ancak randevu sistemi çok karışık, üç kez aramak zorunda kaldım.      BOZULDU  Karışık  Karışık  Olumsuz
Kötü diyemem ama bir daha gelir miyim bilmiyorum.                                   DÜZELDİ  Karışık  Olumsuz  Karışık
Fena değildi.                                                                   hâlâ yanlış  Karışık   Olumlu   Olumlu
Harika bir deneyimdi, sadece üç hafta bekledim.                            

In [4]:
# Sonuç: Küçük ve karışık bir iyileşme; farkların her biri bir iki metnin değişmesinden geliyor.
# - Duygu: +2 puan (200). Düzeldi: "Kötü diyemem..." (çekingen kuralı işe yaradı).
#   Bozuldu: "Arıza çözüldü ancak..." (Prompt 1'de de tutarsızdı). Hâlâ yanlış: ironi ("Harika bir deneyimdi..."),
#   "Fena değildi" (çekingen kuralı uygulanmadı), çift olumsuz (Olumsuz → Karışık, yaklaştı ama yanlış).
# - Konu: 200'de −1, 40'ta +5. "Danışman bey" artık personel içeriyor; ama genel ifadelerden iş_kalitesi çıkarma devam ediyor.
# → Prompt, görev tanımından gelen sorunları kısmen düzeltti; anlama gerektiren hatalar (ironi, genel ifadeden konu çıkarma) düzelmedi.